# Synthetic Single-Cell Population Transport

Task on the linear / bifurcation / tree Splatter benchmarks: predict the withheld
middle pseudotime window (bin 2, `[0.4, 0.6)` of every path) from its two neighbours
(bin 1 → bin 3) with a projector trained on the `holdout_mid` variant, and score the
predicted population against the withheld clean `CellMeans` embedding.

## 0. Setup

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

from IPython.display import HTML, Image, Markdown, display


def _find_repo_root() -> Path:
    here = Path.cwd().resolve()
    for candidate in (here, here.parent, here.parent.parent):
        if (candidate / "experiments" / "toy_manifolds").is_dir():
            return candidate
    raise RuntimeError(
        "Could not locate the repository root (expected experiments/toy_manifolds/)."
    )


REPO_ROOT = _find_repo_root()

for _p in (REPO_ROOT, REPO_ROOT / "src"):
    if str(_p) not in sys.path:
        sys.path.insert(0, str(_p))

from notebooks.helpers import sc_transport as tr
from notebooks.helpers.style import DPI, apply_paper_style


def show_fig(path: Path) -> None:
    """Display the PNG sibling inline (savefig writes both PNG and SVG)."""
    print("saved:", path)
    display(Image(filename=str(path.with_suffix(".png"))))


apply_paper_style()
print(f"REPO_ROOT  = {REPO_ROOT}")
print(f"Figure DPI = {DPI}")


In [ ]:
RESULTS_ROOT_NAME = "synthetic_data"  # "synthetic_data" for the paper runs
RESULTS_ROOT = REPO_ROOT / "results" / RESULTS_ROOT_NAME
OUT_DIR = REPO_ROOT / "notebooks" / "figures" / "sc_transport"

PREFIX = tr.DEFAULT_PREFIX
SEED_PATTERN = tr.DEFAULT_PATTERN
PROTOCOL = tr.DEFAULT_PROTOCOL
SCENARIOS = ["linear", "bifurcation", "tree"]
METHODS = list(tr.DEFAULT_METHODS)
METHODS_OVERVIEW = list(tr.OVERVIEW_METHODS)
PROTOCOL_OVERVIEW = tr.OVERVIEW_PROTOCOL

OUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Results root: {RESULTS_ROOT}")
print(f"Saving figures under: {OUT_DIR}\n")

RUNS: dict[str, list[tuple[str, str]]] = {}
for scenario in SCENARIOS:
    RUNS[scenario] = tr.find_protocol_runs(
        RESULTS_ROOT, scenario, PROTOCOL, prefix=PREFIX
    )
    seeds = ", ".join(label for _, label in RUNS[scenario]) or "none"
    print(f"  {scenario:12s} {len(RUNS[scenario])} runs ({seeds})")


## 4. Summary table

In [ ]:
if not any(RUNS.values()):
    print("No transport runs yet; skipping the summary table.")
else:
    table_rows = tr.collect_seed_metrics(
        RESULTS_ROOT,
        SCENARIOS,
        protocol=PROTOCOL_OVERVIEW,
        methods=METHODS_OVERVIEW,
        pattern=SEED_PATTERN,
    )
    display(HTML(tr.format_seed_table_html(table_rows)))
    print(tr.format_seed_table_latex(table_rows))


## 5. Summary figure

In [ ]:
if not any(RUNS.values()):
    print("No transport runs yet; skipping the overview figure.")
else:
    path, info = tr.fig_metrics_overview(
        OUT_DIR, RESULTS_ROOT, SCENARIOS, protocol=PROTOCOL_OVERVIEW,
        methods=METHODS_OVERVIEW, prefix=PREFIX,
    )
    show_fig(path)
    for scenario in SCENARIOS:
        for metric in ("W1", "gt_tree_dist_mean"):
            panel = info["panels"][f"{metric}:{scenario}"]
            print(
                f"  {scenario:12s} {metric:24s} "
                + " | ".join(
                    f"{m}: {v:.5f} ± {panel['ci95'][m]:.5f}"
                    for m, v in panel["mean"].items()
                )
            )
